# English to Bangla Dataset Translation

This notebook translates English datasets to Bangla (Bengali) using Facebook's NLLB-200-distilled-600M model.

**Datasets:**
- `1B_Model_Low_Reasoning_Data.csv` (reasoning examples)
- `chatbot_conversations.csv` (multi-turn chat)
- `benchmark_llm_reasoning.jsonl` (math benchmark)

**Runtime:** Enable GPU (T4 x2) for fastest translation.

In [ ]:
# Install dependencies
!pip install -q transformers sentencepiece torch pandas tqdm

In [ ]:
import os
import csv
import json
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm import tqdm

# Verify GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Load NLLB-200 model
model_name = "facebook/nllb-200-distilled-600M"
print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = model.to(device)
model.eval()
print("Model loaded successfully!")

In [ ]:
def translate_batch(texts, batch_size=32, max_length=512):
    """Translate a list of texts to Bangla."""
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids("ben_Beng"),
                max_length=max_length,
                num_beams=4,
            )
        translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        results.extend(translations)
    return results

In [ ]:
# Test translation
test_texts = [
    "Hello, how are you?",
    "What is machine learning?",
    "Solve: 2 + 2 = ?"
]
translations = translate_batch(test_texts)
for orig, trans in zip(test_texts, translations):
    print(f"EN: {orig}")
    print(f"BN: {trans}")
    print()

In [ ]:
# Download datasets from Kaggle - USE KAGGLE SECRETS, DO NOT HARDCODE KEYS
# In Kaggle: Settings -> Add-ons -> Secrets -> Add KAGGLE_USERNAME + KAGGLE_KEY
# Or use Kaggle API token via environment (auto-provided in Kaggle notebooks)
!pip install -q kaggle
import os
# Prefer Kaggle Secrets / env vars - hardcoding keys is insecure and rotated
if not os.environ.get("KAGGLE_USERNAME") or not os.environ.get("KAGGLE_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
    except Exception as e:
        print(f"Set KAGGLE_USERNAME/KAGGLE_KEY in Kaggle Secrets. Using env fallback. Error: {e}")
        # Fallback: Kaggle notebooks auto-provide these when internet enabled
        pass

# Download datasets
!kaggle datasets download -d vishesh1412/chatbot-conversations --unzip -p /kaggle/working/data
!kaggle datasets download -d akarshzingare/1b-model-low-reasoning-data --unzip -p /kaggle/working/data
!kaggle datasets download -d akarshzingare/benchmark-llm-reasoning --unzip -p /kaggle/working/data

print("Datasets downloaded!")

In [ ]:
# List downloaded files
!ls -la /kaggle/working/data/

In [ ]:
# Find the actual files
import glob

data_dir = "/kaggle/working/data"
csv_files = glob.glob(os.path.join(data_dir, "**/*.csv"), recursive=True)
jsonl_files = glob.glob(os.path.join(data_dir, "**/*.jsonl"), recursive=True)
print("CSV files:", csv_files)
print("JSONL files:", jsonl_files)

In [ ]:
# Translate Reasoning Dataset
import pandas as pd

reasoning_files = glob.glob(os.path.join(data_dir, "**/*Reasoning*.csv"), recursive=True)
if reasoning_files:
    reasoning_path = reasoning_files[0]
    print(f"Translating: {reasoning_path}")
    df = pd.read_csv(reasoning_path)
    print(f"Rows: {len(df)}")
    
    # Translate in chunks
    chunk_size = 100
    translated_rows = []
    for i in tqdm(range(0, len(df), chunk_size), desc="Reasoning"):
        chunk = df.iloc[i:i+chunk_size]
        prompts = chunk["prompt"].astype(str).tolist()
        answers = chunk["answer"].astype(str).tolist()
        reasoning = chunk["reasoning"].astype(str).tolist()
        
        t_prompts = translate_batch(prompts)
        t_answers = translate_batch(answers)
        t_reasoning = translate_batch(reasoning)
        
        for j in range(len(chunk)):
            translated_rows.append({
                "prompt": t_prompts[j],
                "reasoning": t_reasoning[j],
                "answer": t_answers[j]
            })
    
    # Save translated reasoning
    reasoning_out = pd.DataFrame(translated_rows)
    reasoning_out.to_csv("/kaggle/working/data/1B_Model_Low_Reasoning_Data_bn.csv", index=False)
    print(f"Saved reasoning: {len(translated_rows)} rows")
else:
    print("No reasoning dataset found!")

In [ ]:
# Translate Chat Dataset
chat_files = glob.glob(os.path.join(data_dir, "**/*chatbot*.csv"), recursive=True)
if chat_files:
    chat_path = chat_files[0]
    print(f"Translating: {chat_path}")
    df = pd.read_csv(chat_path)
    print(f"Rows: {len(df)}")
    
    # Translate in chunks
    chunk_size = 1000
    translated_rows = []
    for i in tqdm(range(0, len(df), chunk_size), desc="Chat"):
        chunk = df.iloc[i:i+chunk_size]
        messages = chunk["message"].astype(str).tolist()
        
        t_messages = translate_batch(messages)
        
        for j in range(len(chunk)):
            translated_rows.append({
                "conversation_id": chunk.iloc[j]["conversation_id"],
                "turn": chunk.iloc[j]["turn"],
                "role": chunk.iloc[j]["role"],
                "intent": chunk.iloc[j]["intent"],
                "message": t_messages[j]
            })
    
    # Save translated chat
    chat_out = pd.DataFrame(translated_rows)
    chat_out.to_csv("/kaggle/working/data/chatbot_conversations_bn.csv", index=False)
    print(f"Saved chat: {len(translated_rows)} rows")
else:
    print("No chat dataset found!")

In [ ]:
# Translate Benchmark Dataset
benchmark_files = glob.glob(os.path.join(data_dir, "**/*.jsonl"), recursive=True)
if benchmark_files:
    benchmark_path = benchmark_files[0]
    print(f"Translating: {benchmark_path}")
    
    rows = []
    with open(benchmark_path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line.strip()))
    
    print(f"Rows: {len(rows)}")
    
    # Translate in chunks
    chunk_size = 100
    translated_rows = []
    for i in tqdm(range(0, len(rows), chunk_size), desc="Benchmark"):
        chunk = rows[i:i+chunk_size]
        questions = [r.get("question", "") for r in chunk]
        
        t_questions = translate_batch(questions)
        
        for j, r in enumerate(chunk):
            translated_rows.append({
                "id": r.get("id"),
                "category": r.get("category", ""),
                "question": t_questions[j],
                "answer": r.get("answer"),
                "difficulty": r.get("difficulty"),
                "type": r.get("type", "")
            })
    
    # Save translated benchmark
    with open("/kaggle/working/data/benchmark_llm_reasoning_bn.jsonl", "w", encoding="utf-8") as f:
        for item in translated_rows:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"Saved benchmark: {len(translated_rows)} rows")
else:
    print("No benchmark dataset found!")

In [ ]:
# Verify all translated files
print("=== Translated Files ===")
for f in sorted(glob.glob("/kaggle/working/data/*_bn*")):
    size = os.path.getsize(f) / 1e6
    print(f"{os.path.basename(f)}: {size:.2f} MB")

In [ ]:
# Preview translated data
print("=== Reasoning Sample ===")
if os.path.exists("/kaggle/working/data/1B_Model_Low_Reasoning_Data_bn.csv"):
    df = pd.read_csv("/kaggle/working/data/1B_Model_Low_Reasoning_Data_bn.csv")
    print(df.head(2).to_string())

print("\n=== Chat Sample ===")
if os.path.exists("/kaggle/working/data/chatbot_conversations_bn.csv"):
    df = pd.read_csv("/kaggle/working/data/chatbot_conversations_bn.csv")
    print(df.head(5).to_string())

print("\n=== Benchmark Sample ===")
if os.path.exists("/kaggle/working/data/benchmark_llm_reasoning_bn.jsonl"):
    with open("/kaggle/working/data/benchmark_llm_reasoning_bn.jsonl", "r") as f:
        for i, line in enumerate(f):
            if i >= 3: break
            print(json.loads(line))

In [ ]:
# Create a zip file for download
!cd /kaggle/working/data && zip -j /kaggle/working/bangla_datasets.zip *_bn*
print("\nZip file created: /kaggle/working/bangla_datasets.zip")